In [2]:
import os
import time
import csv
import subprocess
import psutil
import ast
from pathlib import Path

# ----- CONFIGURACIONES A EVALUAR (deben coincidir con la generación previa) -----
MODELOS = [
    "gemini-2.0-flash-lite",
    "gemini-2.0-flash",
    "gemini-2.0-pro-exp-02-05"
]
TEMPERATURAS = [0, 0.5, 1, 2]
PARAM_SETS = [
    {"top_p": 0.1, "top_k": 10},   # Más determinista
    {"top_p": 0.5, "top_k": 30},   # Moderado
    {"top_p": 0.95, "top_k": 100}  # Menos determinista
]

# Límite de tiempo para la ejecución de cada script (en segundos)
TIMEOUT_EXEC = 120

# Directorio base donde se encuentran los archivos generados previamente
BASE_OUTPUT_DIR = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs"
# Directorio base donde se guardará el CSV de resultados
BASE_RESULTS_DIR = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\results"

def measure_complexity(code: str) -> str:
    """
    Mide una complejidad algorítmica simple contando el número de bucles y condicionales.
    Esta medida es una aproximación para fines comparativos.
    """
    try:
        tree = ast.parse(code)
    except Exception:
        return "N/A"
    loops = sum(isinstance(node, (ast.For, ast.While)) for node in ast.walk(tree))
    conditionals = sum(isinstance(node, ast.If) for node in ast.walk(tree))
    return f"loops: {loops}, conditionals: {conditionals}"

def run_script_with_metrics(script_path: str, timeout: int = TIMEOUT_EXEC):
    """
    Ejecuta el script Python en 'script_path' con un límite de tiempo y mide:
      - Tiempo de ejecución
      - Memoria máxima utilizada (en MB)
    Devuelve (resultado, tiempo_ejecución, memoria_maxima_MB).
    """
    start_time = time.time()
    process = psutil.Popen(["python", script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    max_memory = 0
    stdout, stderr = b"", b""
    while True:
        if process.poll() is not None:
            stdout, stderr = process.communicate()
            break
        if time.time() - start_time > timeout:
            process.kill()
            return "TIMEOUT", timeout, max_memory / (1024 * 1024)
        try:
            mem = process.memory_info().rss
            if mem > max_memory:
                max_memory = mem
        except psutil.NoSuchProcess:
            break
        time.sleep(0.1)
    execution_time = time.time() - start_time
    result = stdout.decode().strip() if stdout else stderr.decode().strip()
    return result, execution_time, max_memory / (1024 * 1024)

def main():
    results = []
    # Itera sobre cada combinación de modelo, temperatura y configuración de top_p/top_k
    for model in MODELOS:
        for temp in TEMPERATURAS:
            for params in PARAM_SETS:
                top_p = params["top_p"]
                top_k = params["top_k"]
                # Directorio donde se encuentran los scripts generados para esta configuración
                output_directory = Path(BASE_OUTPUT_DIR) / f"temperature-{temp}" / model / f"top_p-{top_p}_top_k-{top_k}"
                if not output_directory.exists():
                    print(f"No se encontró el directorio {output_directory}")
                    continue
                # Buscar todos los archivos output_*.py ordenados por número (e.g. output_1.py, output_2.py, ...)
                output_files = sorted(output_directory.glob("output_*.py"),
                                      key=lambda f: int(f.stem.split("_")[1]))
                for output_file in output_files:
                    with open(output_file, "r", encoding="utf-8") as f:
                        python_code = f.read()
                    print(f"Evaluando {output_file}")
                    exec_result, exec_time, mem_usage = run_script_with_metrics(str(output_file))
                    algo_complexity = measure_complexity(python_code)
                    true_count = exec_result.count("True") if isinstance(exec_result, str) else 0
                    false_count = exec_result.count("False") if isinstance(exec_result, str) else 0
                    # Extraer el número de problema desde el nombre del archivo (por ejemplo, "output_3.py" → 3)
                    problem_id = int(output_file.stem.split("_")[1])
                    results.append({
                        "ID": problem_id,
                        "model": model,
                        "temperature": temp,
                        "top_p": top_p,
                        "top_k": top_k,
                        "code": python_code,
                        "result": exec_result,
                        "true_count": true_count,
                        "false_count": false_count,
                        "execution_time": round(exec_time, 2),
                        "memory_usage_MB": round(mem_usage, 2),
                        "algorithmic_complexity": algo_complexity
                    })
    # Guarda todos los resultados en un archivo CSV
    results_csv = Path(BASE_RESULTS_DIR) / "results_completo.csv"
    results_csv.parent.mkdir(parents=True, exist_ok=True)
    with open(results_csv, mode="w", newline="", encoding="utf-8") as csvfile:
        fieldnames = [
            "ID", "model", "temperature", "top_p", "top_k",
            "code", "result", "true_count", "false_count",
            "execution_time", "memory_usage_MB", "algorithmic_complexity"
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"CSV de resultados guardado en: {results_csv}")

if __name__ == "__main__":
    main()


Evaluando C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs\temperature-0\gemini-2.0-flash-lite\top_p-0.1_top_k-10\output_1.py
Evaluando C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs\temperature-0\gemini-2.0-flash-lite\top_p-0.1_top_k-10\output_2.py
Evaluando C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs\temperature-0\gemini-2.0-flash-lite\top_p-0.1_top_k-10\output_3.py
Evaluando C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs\temperature-0\gemini-2.0-flash-lite\top_p-0.1_top_k-10\output_4.py
Evaluando C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs\temperature-0\gemini-2.0-flash-lite\top_p-0.1_top_k-10\output_5.py
Evaluando C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs\temperature-0\gemini-2.0-flash-lite\top_p-0.5_top_k-30\output_1.py
Evaluando C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\outputs\temperature-0\gemini-2.0-flash-lite\top_p-0.5_top_k-30\output_2.py
Evalua